# Final mini-project - L2 error type (AutoErrorAnalyzer)

*Track: `l2_errors`.* Classify each L2 English sentence into Grammatical / Lexical / Mechanical / No error. Build `l2_errors_pool.json` and drop it in `data/gold/` first.

This notebook imports the given helpers from `../scripts` (`pipeline.py` for the plumbing, `evaluate.py` for the scoring). You edit the **CONFIG cell**, your **prompt file**, and the **QC cell** - nothing else.

**Pipeline:** sample a balanced ~40-item gold subset -> QC/adjudicate -> baseline prompt -> iterate 2 few-shot rounds (P/R/F1 + confusion matrix each round) -> error analysis -> export a one-page report.

## Setup - run this first

Gets the code on the path and starts the LLM backend. In Colab, uncomment a clone block first (see the comments in the cell).

In [ ]:
# ------------------------------------------------------------------
# SETUP - run me first.
# ------------------------------------------------------------------
# In Google Colab, first get the template code onto the machine by UNCOMMENTING
# one of the two blocks below, then run the whole cell.

# --- Colab Option A: clone into your Google Drive (persists across sessions) ---
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive
# ![ -d lda2-final-template ] || git clone https://github.com/egumasa/lda2-final-template.git lda2-final-template
# %cd /content/drive/MyDrive/lda2-final-template/notebooks

# --- Colab Option B: quick, ephemeral clone (changes lost on runtime reset) ---
# !git clone https://github.com/egumasa/lda2-final-template.git
# %cd lda2-final-template/notebooks

# Put the scripts/ folder on the import path. Works locally AND in Colab
# (after the %cd above), because notebooks/ and scripts/ are side by side.
import sys
sys.path.append("../scripts")
import random

from pipeline import *
from evaluate import *

generate_text, backend = make_backend()
print("LLM backend:", backend)

## Config

In [ ]:
# ------------------------------------------------------------------
# CONFIG - the only cell you normally change.
# ------------------------------------------------------------------
TRACK       = "l2_errors"
SEED        = 42          # change per group so each draws a different subset
N_PER_CLASS = 10          # items to try to sample per label
OUT_DIR     = "../outputs"   # where the report and CSV are written

# Where the pool of labeled items lives (relative to this notebook in notebooks/).
POOL_PATH = "../data/gold/" + TRACK + "_pool.json"
# If you house your pool in Google Drive instead, point at it directly, e.g.:
# POOL_PATH = "/content/drive/MyDrive/lda2/gold/" + TRACK + "_pool.json"

# Your prompt lives in prompts/<track>.txt - edit that FILE to iterate (not this line).
PROMPT_FILE = "../prompts/" + TRACK + ".txt"

## Step 1 - Sample a balanced gold subset

`sample_pool` draws up to *N* items per label. Rare classes yield fewer - a property of the data.

In [ ]:
pool = load_gold(POOL_PATH)
gold = sample_pool(pool, N_PER_CLASS, SEED)
labels = label_set(gold)
print("Labels:", labels)

## Step 2 - QC / adjudicate

Each member independently re-checks part of the subset, then you compare and resolve. `agreement()` reports percent agreement and Cohen's kappa - the number that says whether the *scheme* is crisp or fuzzy. The cell below **simulates** a second annotator; replace `annot_b` with a teammate's real labels.

In [ ]:
# Annotator A = the published gold label for each item.
annot_a = []
for item in gold:
    annot_a.append(item["label"])

# Annotator B = a SIMULATED second annotator. We copy annotator A, but for about
# 15% of the items we deliberately choose a DIFFERENT label, to imitate a real
# disagreement between two people. In a real group, REPLACE annot_b with a
# teammate's own independent labels for the same items.
random_generator = random.Random(SEED + 1)
annot_b = []
for label_a in annot_a:
    if random_generator.random() < 0.15:
        other_labels = []
        for possible_label in labels:
            if possible_label != label_a:
                other_labels.append(possible_label)
        annot_b.append(random_generator.choice(other_labels))
    else:
        annot_b.append(label_a)

# Compare the two annotators: percent agreement + Cohen's kappa.
agreement(annot_a, annot_b)

# List the items where they disagree - these are what a real group discusses.
disagreements = []
for item, label_a, label_b in zip(gold, annot_a, annot_b):
    if label_a != label_b:
        disagreements.append((item["id"], label_a, label_b))
print(len(disagreements), "items to adjudicate. First few:", disagreements[:3])

## Step 3 - Baseline prompt (iteration 0)

Zero-shot. The prompt is loaded from your track's file in `prompts/`; `{text}` is where each sentence is slotted in.

In [ ]:
f1_by_round = {}

PROMPT = load_prompt(PROMPT_FILE)
print(PROMPT)
pred0 = run_prompt(PROMPT, gold, labels, generate_text)
f1_by_round["iter0 zero-shot"] = evaluate(gold, pred0, labels, TRACK + " - iter0")

## Step 4 - Iterate the prompt (few-shot rounds 1-2)

`build_fewshot` puts labeled examples from the **pool, never the gold set**, in front of the prompt. Round 1 adds one example per class; round 2 adds two.

In [ ]:
prompt1 = build_fewshot(PROMPT, pool, gold, labels, 1, SEED)
pred1 = run_prompt(prompt1, gold, labels, generate_text)
f1_by_round["iter1 few-shot x1"] = evaluate(gold, pred1, labels, TRACK + " - iter1")

In [ ]:
prompt2 = build_fewshot(PROMPT, pool, gold, labels, 2, SEED)
pred2 = run_prompt(prompt2, gold, labels, generate_text)
f1_by_round["iter2 few-shot x2"] = evaluate(gold, pred2, labels, TRACK + " - iter2")

print("Macro-F1 by round:")
for round_name in f1_by_round:
    print("  ", round_name, ":", format(f1_by_round[round_name], ".3f"))

## Step 5 - Error analysis

For each miss: is the **gold** defensible, or a genuinely borderline item? *"Model's fault or the scheme's?"* Uses the best round's predictions.

In [ ]:
# Find which round had the highest macro-F1.
best_round = None
best_score = -1.0
for round_name in f1_by_round:
    score = f1_by_round[round_name]
    if score > best_score:
        best_score = score
        best_round = round_name

# Pick the predictions that came from that best round.
if best_round == "iter0 zero-shot":
    best_predictions = pred0
elif best_round == "iter1 few-shot x1":
    best_predictions = pred1
else:
    best_predictions = pred2

print("Best round:", best_round, "with macro-F1", format(best_score, ".3f"))
show_errors(gold, best_predictions)

## Step 6 - Export the one-page report

In [ ]:
export_results(TRACK, gold, best_predictions, f1_by_round, OUT_DIR)

---
Full loop done: **sample -> QC -> baseline -> iterate -> error analysis -> report.** Open `outputs/l2_errors_report.md` and fill in the QC and error-analysis prose.